In [ ]:
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import regex
from tqdm.auto import tqdm

from src.processor import LogProcessor
from src.utils import (
    getFilesByDate,
    guardarExcel,
    parallelizeFunction,
    time2localtime,
)

In [ ]:
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "BAJA",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ALTA",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    # "Stopped": "STOP",
    # "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "MANIOBRALLEGADA",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "EXIT",
            "MANIOBRASALIDA",
            "MANIOBRA",
        ]
    )
}

### Cambios Sentido
- XSIV: maniobras antes/después de salida/llegada
- MIE: Cambio de paridad en un mismo enclavamiento

#### XSIV

In [ ]:
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "BAJA",
    "End": "FIN",
    # "Entry": "ENTRY",
    # "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ALTA",
    # "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    # "Stopped": "STOP",
    # "TrackingLost": "LOST_TRACK",
}

log_processor = LogProcessor()

w_logs = getFilesByDate(
    Path(r"C:\Users\jose.espinosa\Documents\Data\xsiv\PRO"),
    "2024-12-01",
    "2024-12-10",
)
fnames, full_days = list(zip(*w_logs))
days = f"{full_days[0].strftime('%Y-%m-%d')} - {full_days[-1].strftime('%Y-%m-%d')}"
xsiv_df = log_processor.loadFilesLogs(fnames, "xsiv", train_types, days)

In [ ]:
# Índices de trenes con maniobra y llegada/salida/aproximación en una misma estación
xsiv_df_split = (
    xsiv_df[["Movimiento", "FechaOrigen", "NTécnico", "Código"]]
    .reset_index()
    .groupby(by=["FechaOrigen", "NTécnico", "Código"])
    .agg(set)
    .reset_index()
)


xsiv_df_split = xsiv_df_split.loc[
    xsiv_df_split["Movimiento"].apply(
        lambda x: "MANIOBRA" in x
        and any([mov in x for mov in ["LLEGADA", "SALIDA", "APROXIMACIÓN"]])
    ),
    "index",
].apply(list)

In [ ]:
xsiv_errors = []
minute_thr = 1
use_xsiv_df = xsiv_df.loc[xsiv_df_split.sum()].copy()
for dup in xsiv_df_split.values:
    xsiv_error = use_xsiv_df.loc[dup].sort_values(by="Fecha")
    maneuver = xsiv_error[xsiv_error["Movimiento"] == "MANIOBRA"]
    first_maneuver = maneuver.iloc[0]
    last_maneuver = maneuver.iloc[-1]
    first_in = xsiv_error[xsiv_error["Movimiento"].isin(["LLEGADA", "APROXIMACIÓN"])]
    first_out = xsiv_error[xsiv_error["Movimiento"].isin(["SALIDA"])]
    elimination = xsiv_error[xsiv_error["Movimiento"].isin(["BAJA", "FIN"])]
    if not first_in.empty:
        first_in = first_in["Fecha"].iloc[0]
        if first_maneuver["Fecha"] < first_in + timedelta(minutes=minute_thr):
            xsiv_errors.append(
                xsiv_error[["FechaOrigen", "NTécnico", "Nombre", "Código"]].iloc[0]
            )
    if not first_out.empty:
        first_out = first_out["Fecha"].iloc[0]
        if last_maneuver["Fecha"] > first_out - timedelta(minutes=minute_thr):
            if (
                not elimination.empty
                and elimination["Fecha"].iloc[0] < last_maneuver["Fecha"]
                and last_maneuver["Estado"] == "FINISHED"
            ):
                continue
            xsiv_errors.append(
                xsiv_error[["FechaOrigen", "NTécnico", "Nombre", "Código"]].iloc[0]
            )
xsiv_errors = (
    pd.DataFrame(xsiv_errors)
    .drop_duplicates()
    .sort_values(by=["FechaOrigen", "Código", "NTécnico"])
    .reset_index(drop=True)
)

In [ ]:
display(xsiv_errors.head())
display(xsiv_errors["Nombre"].value_counts().head())

In [ ]:
# Muestra de algunos de los potenciales errores
count = 0


for _, e in xsiv_errors.iterrows():
    ex = use_xsiv_df[
        (use_xsiv_df["FechaOrigen"] == e["FechaOrigen"])
        & (use_xsiv_df["NTécnico"] == e["NTécnico"])
        & (use_xsiv_df["Código"] == e["Código"])
    ].sort_values(by="Fecha")

    if ex.loc[ex["Movimiento"] == "MANIOBRA", "Estado"].iloc[-1] == "FINISHED":
        continue

    display(ex)

    count += 1

    if count == 10:

        break

#### MIE

In [ ]:
start_date = "2025-03-10"
end_date = "2025-03-10"

##### Cambio paridad circuito

In [ ]:
dir_mies = Path(r"C:\Users\jose.espinosa\Documents\Data\mie_mse")

w_logs = getFilesByDate(dir_mies, start_date, end_date)
fnames, full_days = list(zip(*w_logs))
days = f"{full_days[0].strftime('%Y-%m-%d')} - {full_days[-1].strftime('%Y-%m-%d')}"
log_processor = LogProcessor()
mie_df = log_processor.loadFilesLogs(
    fnames, "mie_mse", load="tren", mtype=["ocupa", "libera"], days=days
)

In [ ]:
mie_df.head()

In [ ]:
parity = (
    mie_df.dropna(subset=["MIE", "NTécnico", "Nombre", "Elemento", "Fecha"])
    .drop_duplicates()
    .sort_values(by=["MIE", "NTécnico", "Nombre", "Elemento", "Fecha"])
)
parity_split = np.split(
    parity,
    np.where(
        (~parity["MIE"].eq(parity["MIE"].shift()))
        | (~parity["NTécnico"].eq(parity["NTécnico"].shift()))
        | (~parity["Nombre"].eq(parity["Nombre"].shift()))
        | (~parity["Elemento"].eq(parity["Elemento"].shift()))
        | np.invert(parity["Fecha"] < parity["Fecha"].shift() + timedelta(hours=1))
    )[0][1:],
)
parity_split_filt = [el for el in parity_split if len(set(el["Sentido"])) > 1]

In [ ]:
# parity_split_filt = [
#     sp
#     for el in parity_split_filt
#     for sp in np.split(
#         el,
#         np.where(np.invert(el["Fecha"] < el["Fecha"].shift() + timedelta(hours=1)))[0][
#             1:
#         ],
#     )
# ]

In [ ]:
len(parity_split_filt)

In [ ]:
parity_split_filt[0]

##### Cambio paridad enclavamiento

In [ ]:
dir_mies = Path(r"C:\Users\jose.espinosa\Documents\Data\mie_mse")

w_logs = getFilesByDate(dir_mies, start_date, end_date)
fnames, full_days = list(zip(*w_logs))
days = f"{full_days[0].strftime('%Y-%m-%d')} - {full_days[-1].strftime('%Y-%m-%d')}"
log_processor = LogProcessor()
mie_df = log_processor.loadFilesLogs(
    fnames, "mie_mse", load="tren", mtype=["ocupa"], days=days
)

In [ ]:
# mie_df[
#     (mie_df["MIE"] == "/opt/appl/logs/mse/mie_mse_objectsBILBAO.log")
#     & (mie_df["Elemento"].isin(["E4", "E2"]))
#     & (mie_df["Mnemónico"] == "AI")
#     & (mie_df["Fecha"] > pd.to_datetime("2025-02-02 05:00:00"))
#     & (mie_df["Fecha"] < pd.to_datetime("2025-02-02 12:00:00"))
# ].sort_values(by="Fecha")

In [ ]:
# mie_df[(mie_df["NTécnico"] == "26511") & (mie_df["Mnemónico"] == "AI")].sort_values(
#     by="Fecha"
# )

In [ ]:
parity = mie_df.drop_duplicates().sort_values(by=["MIE", "NTécnico", "Nombre", "Fecha"])
parity_split = np.split(
    parity,
    np.where(
        (~parity["MIE"].eq(parity["MIE"].shift()))
        | (~parity["NTécnico"].eq(parity["NTécnico"].shift()))
        | (~parity["Nombre"].eq(parity["Nombre"].shift()))
    )[0][1:],
)
parity_split = [el for el in parity_split if len(set(el["Sentido"])) > 1]
# mie_errors = []
# for el in parity_split:
#     if len(set(el["element_count"])) > 1:
#         mie_errors.append(el)
# dup = parity[["mie", "tren", "mnemonic"]].duplicated(keep=False)
# parity = parity[dup].sort_values(by=["mie", "tren", "mnemonic", "parity"])
# parity.columns = ["_".join([el for el in c if el]) for c in parity.columns]

In [ ]:
parity_split[0].head()

In [ ]:
state_changes = []
for el in parity_split:
    mie_error = np.split(
        el,
        np.where((~el["Sentido"].eq(el["Sentido"].shift())))[0][1:],
    )
    for i in range(len(mie_error) - 1):
        orig_circuit = mie_error[i].iloc[-1]
        dest_circuit = mie_error[i + 1].iloc[0]
        if dest_circuit["Fecha"] - orig_circuit["Fecha"] > timedelta(minutes=10):
            continue
        state_changes.append(
            {
                "MIE": orig_circuit["MIE"],
                "Fecha": orig_circuit["Fecha"],  # .strftime("%Y-%m-%d"),
                "NTécnico": orig_circuit["NTécnico"],
                "Mnemónico": orig_circuit["Mnemónico"],
                "ElementoOrigen": orig_circuit["Elemento"],
                "ParidadOrigen": orig_circuit["Sentido"],
                "ElementoDestino": dest_circuit["Elemento"],
                "ParidadDestino": dest_circuit["Sentido"],
            }
        )
state_changes = pd.DataFrame(state_changes)[
    [
        "MIE",
        "Mnemónico",
        "Fecha",
        "NTécnico",
        "ElementoOrigen",
        "ParidadOrigen",
        "ElementoDestino",
        "ParidadDestino",
    ]
].sort_values(by=["MIE", "Fecha", "Mnemónico", "NTécnico"])
state_changes = state_changes[
    ~(state_changes["ElementoOrigen"] == state_changes["ElementoDestino"])
].reset_index(drop=True)

In [ ]:
# state_changes[state_changes["NTécnico"] == "SPAÑA"]

In [ ]:
resumen = (
    state_changes[
        [
            "MIE",
            "Mnemónico",
            "ElementoOrigen",
            "ParidadOrigen",
            "ElementoDestino",
            "ParidadDestino",
        ]
    ]
    .groupby(
        [
            "MIE",
            "Mnemónico",
            "ElementoOrigen",
            "ParidadOrigen",
            "ElementoDestino",
            "ParidadDestino",
        ],
        as_index=False,
        dropna=False,
    )
    .size()
    .sort_values(by=["size", "MIE", "Mnemónico", "ElementoOrigen"], ascending=False)
    .rename(columns={"size": "ocurrencias"})
    .reset_index(drop=True)
)

In [ ]:
# state_changes[state_changes["NTécnico"] == "26028"]

In [ ]:
resumen.head()

In [ ]:
guardarExcel(
    resumen,
    "Errores MIE 2025-03-10.xlsx",
    sheet_name="Resumen",
    append_sheet=False,
)
for mie_df_s in np.split(
    state_changes,
    np.where((~state_changes["MIE"].eq(state_changes["MIE"].shift())))[0][1:],
):
    guardarExcel(
        mie_df_s.reset_index(drop=True),
        "Errores MIE 2025-03-10.xlsx",
        sheet_name=regex.search(
            r"(?<=_objects).+(?=\.log)", mie_df_s["MIE"].iloc[0]
        ).group(),
        append_sheet=True,
    )

In [ ]:
resumen

In [ ]:
# parity = (
#     mie_df.groupby(["mie", "tren", "mnemonic", "parity"])
#     .agg({"element": [list, "count"]})
#     .reset_index()
# )
# dup = parity[["mie", "tren", "mnemonic"]].duplicated(keep=False)
# parity = parity[dup].sort_values(by=["mie", "tren", "mnemonic", "parity"])
# parity.columns = ["_".join([el for el in c if el]) for c in parity.columns]

In [ ]:
parity_split = np.split(
    parity,
    np.where(
        (~parity["mie"].eq(parity["mie"].shift()))
        | (~parity["tren"].eq(parity["tren"].shift()))
        | (~parity["mnemonic"].eq(parity["mnemonic"].shift()))
    )[0][1:],
)
mie_errors = []
for el in parity_split:
    if len(set(el["element_count"])) > 1:
        mie_errors.append(el)

In [ ]:
mie_errors = pd.concat(mie_errors).sort_values(by=["mie", "tren", "mnemonic", "parity"])
errors_split = np.split(
    mie_errors,
    np.where((~mie_errors["mie"].eq(mie_errors["mie"].shift())))[0][1:],
)

In [ ]:
for mie_df_s in errors_split:
    guardarExcel(
        mie_df_s[["mie", "mnemonic", "tren", "parity", "element_list", "element_count"]]
        .sort_values(by=["mie", "mnemonic", "tren", "parity"])
        .reset_index(drop=True),
        "Errores MIE.xlsx",
        sheet_name=mie_df_s["mie"].iloc[0],
        append_sheet=True,
    )

In [ ]:
mie_errors = mie_df_s[mie_df_s["parity"].apply(len) > 1]
mie_errors[mie_errors[["mie", "mnemonic"]].duplicated(keep=False)][
    ["mie", "mnemonic", "parity"]
].groupby(["mie", "mnemonic"]).agg("sum").reset_index().sort_values(
    by=["mie", "mnemonic"]
)